## Étape 6 — Calcul du SMR (Standardised Mortality Ratio)

**SMR = Décès observés / Décès attendus** selon la table TH/TF 00-02.

| SMR | Interprétation |
|---|---|
| < 0.90 | Portefeuille moins mortel (sélection favorable) |
| 0.90 – 1.10 | Conforme à la référence |
| > 1.10 | Sur-mortalité |

In [ ]:
table['q_x_ref'] = table['age'].map(lambda a: qx_ref(a))
table['deces_attendus'] = table['E_x'] * table['q_x_ref']
D_obs = table['D_x'].sum()
D_att = table['deces_attendus'].sum()
SMR   = D_obs / D_att if D_att > 0 else np.nan
with np.errstate(divide='ignore', invalid='ignore'):
    table['SMR_local'] = np.where(
        table['deces_attendus'] > 0,
        table['D_x'] / table['deces_attendus'], np.nan)
ci_half = 1.96 * np.sqrt(D_obs) / D_att if D_att > 0 else np.nan
print('Deces observes   : {:,}'.format(int(D_obs)))
print('Deces attendus   : {:.1f}'.format(D_att))
print('SMR global       : {:.4f}'.format(SMR))
print('IC 95%           : [{:.4f}, {:.4f}]'.format(SMR-ci_half, SMR+ci_half))
print()
if SMR < 0.90:
    print('=> Portefeuille moins mortel que la reference TH 00-02 (selection favorable).')
elif SMR > 1.10:
    print('=> Portefeuille plus mortel que la reference TH 00-02 (sur-mortalite).')
else:
    print('=> Mortalite conforme a la table de reference TH 00-02.')
print()
print('SMR par tranche d age :')
for ds in range(30, 90, 10):
    m = (table['age'] >= ds) & (table['age'] < ds+10)
    d_o = table.loc[m,'D_x'].sum()
    d_a = table.loc[m,'deces_attendus'].sum()
    smr_d = d_o/d_a if d_a > 0 else float('nan')
    print('  {:}-{:} ans : SMR={:.3f}  (obs={:}, att={:.1f})'.format(ds, ds+9, smr_d, int(d_o), d_a))
